# Employee Salary Prediction
A beginner-friendly machine-learning workflow. Run all cells from top to bottom.

## Step 1 — Import libraries

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

BASE_DIR = Path.cwd()
SOURCE_PATH = BASE_DIR.parent / 'enhanced_salary_prediction_dataset_1500.xlsx'
DATA_PATH = BASE_DIR / 'data.xlsx'
MODEL_PATH = BASE_DIR / 'salary_model.pkl'


## Step 2 — Load and prepare dataset
The provided workbook calls daily hours `Working_Hours`. We rename it and calculate weekly hours as daily hours × 5.

In [ ]:
source = pd.read_excel(SOURCE_PATH)
data = source[['Position', 'Experience_Years', 'Working_Hours', 'Salary']].copy()
data = data.rename(columns={'Working_Hours': 'Working_Hours_Per_Day'})
data['Weekly_Hours'] = (data['Working_Hours_Per_Day'] * 5).round(1)
data.to_excel(DATA_PATH, index=False)

print(data.head())
print('Rows and columns:', data.shape)
print('Columns:', data.columns.tolist())
data.info()

## Step 3 — Data cleaning

In [ ]:
print('Missing values:\n', data.isnull().sum())
print('Duplicate records:', data.duplicated().sum())

numeric_columns = ['Experience_Years', 'Working_Hours_Per_Day', 'Weekly_Hours', 'Salary']
for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

data = data.dropna().drop_duplicates()
data = data[(data['Salary'] > 0) & (data['Experience_Years'] >= 0) &
            (data['Working_Hours_Per_Day'] > 0) & (data['Working_Hours_Per_Day'] <= 24) &
            (data['Weekly_Hours'] > 0)]
data.to_excel(DATA_PATH, index=False)
print('Cleaned dataset shape:', data.shape)

## Step 4 — Data visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].scatter(data['Experience_Years'], data['Salary'], alpha=0.55, color='#183b5b')
axes[0, 0].set(title='Experience vs Salary', xlabel='Experience (years)', ylabel='Salary (INR)')
axes[0, 1].scatter(data['Weekly_Hours'], data['Salary'], alpha=0.55, color='#a16d2d')
axes[0, 1].set(title='Weekly Hours vs Salary', xlabel='Weekly hours', ylabel='Salary (INR)')
axes[1, 0].scatter(data['Working_Hours_Per_Day'], data['Salary'], alpha=0.55, color='#38755b')
axes[1, 0].set(title='Working Hours per Day vs Salary', xlabel='Daily hours', ylabel='Salary (INR)')
average_salary = data.groupby('Position')['Salary'].mean().sort_values()
axes[1, 1].barh(average_salary.index, average_salary.values, color='#183b5b')
axes[1, 1].set(title='Average Salary by Position', xlabel='Salary (INR)')
for axis in axes.flat:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Steps 5–7 — Preprocess, split, and train
Linear Regression is used because it is simple to explain and provides a clear baseline. The pipeline avoids data leakage by learning position encoding only from training data.

In [ ]:
features = ['Position', 'Experience_Years', 'Working_Hours_Per_Day', 'Weekly_Hours']
X = data[features]
y = data['Salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

preprocessor = ColumnTransformer([
    ('position', OneHotEncoder(handle_unknown='ignore'), ['Position']),
    ('numbers', 'passthrough', ['Experience_Years', 'Working_Hours_Per_Day', 'Weekly_Hours'])
])
model = Pipeline([('preprocessor', preprocessor), ('regression', LinearRegression())])
model.fit(X_train, y_train)
print('Training records:', len(X_train), '| Testing records:', len(X_test))

## Steps 8–9 — Evaluate and save model

In [ ]:
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(f'MAE:  INR {mean_absolute_error(y_test, predictions):,.2f}')
print(f'MSE:  {mse:,.2f}')
print(f'RMSE: INR {np.sqrt(mse):,.2f}')
print(f'R² Score: {r2_score(y_test, predictions):.3f}')

joblib.dump(model, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')